Install required dependencies

In [1]:
%pip install langchain_community unstructured azure-identity langchain_openai azure-search-documents

Import environment variables

In [2]:
from google.colab import userdata
from google.colab import auth

AZURE_OPEN_API_ENDPOINT = userdata.get("AZURE_OPEN_API_ENDPOINT")
AZURE_OPEN_API_KEY = userdata.get("AZURE_OPEN_API_KEY")

VECTOR_SEARCH_ENDPOINT = userdata.get("VECTOR_SEARCH_ENDPOINT")
VECTOR_SEARCH_KEY = userdata.get("VECTOR_SEARCH_KEY")

auth.authenticate_user()

Fetch the data

In [3]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-900',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-300',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/dp-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-731',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-730',
]

loader = UnstructuredURLLoader(urls)

documents = loader.load()

print(len(documents))

7


Split the data into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

21


Store the data into vector database

In [5]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="text-embedding-3-small",
    openai_api_version="2023-05-15",
)

vector_store = AzureSearch(
    azure_search_endpoint=VECTOR_SEARCH_ENDPOINT,
    azure_search_key=VECTOR_SEARCH_KEY,
    index_name='consine-similarity-demo',
    embedding_function=embeddings.embed_query
)

vector_store.add_documents(chunks)

['MzY3ZGZhMmUtMmE4OS00ZTQzLWIxZTYtNTk5NGVlMDljYmFm',
 'MTIwNjM4YmUtZmRkZC00MzhlLWI5N2EtZjM3NmIyNzE2NDZi',
 'MGZhNGYzYmQtNjNmYS00N2I0LWFiYWMtZDQxNDViMDA5NDRh',
 'MjA1ZmI0ZDctZDI1Mi00OTM4LTg2MzgtN2U0MzgxNDE1ZDc4',
 'MTU5ZGQyZmQtZTViYi00NGJmLTk4MjgtYmVkM2E0ODY2YmQ1',
 'YTU0NTg2ZWMtNzQyMS00YTQxLWEwNTItYTYxY2JmNmI5NWQ3',
 'NGE2Yzg1ZjItOTZiZi00YmRiLThhYTctM2VhZmYxOTEzNTFi',
 'ZjlmMmFmZDQtYjA1ZC00MmJhLWE1ZWEtMTA4MzRmYTlhMzM0',
 'ODMwN2U2MWItNGUyYi00ZjI3LWJiNTYtMjJkNDk5NWIxMWQ1',
 'MGNmODcxYmUtMjQ2Yi00YzNmLWJlNzItOTA3YWUzNGY5OWIx',
 'YzIxMjk2ODItMjA1NC00ODhhLTgwZTctYmMzZjU5YzcxODVj',
 'MWYxMjM5NTItMDljNy00MzBjLWI4NDEtMTVjYjExOWExMjUw',
 'YzZkMDE5YTEtMGM1Yy00NTRlLWI0OTYtNjBkNzZlMDFmYWVh',
 'NTQ3NjIzYTctZmFhMC00MTEzLWJkNzEtYmY3NGE4NGVjYjVm',
 'MjMwNGI2YTgtOTYwNy00MTEyLWIzYzYtNWIwZDg4NTlkMDlj',
 'ZDI4ZWI0N2YtZmU1My00NjJmLTliODUtMmJlYzAxNDE5MmNl',
 'NTFhNmYwYWYtMzBkNC00NTI5LTk1MDItMmY0YjgwM2ZkZDIy',
 'ZjFiNjMyZTEtYTEyZi00ZWExLWI4ZDctYWM3Y2NjNmJlM2Yw',
 'MDE1ZGUzNWUtYTVkNy00YmZmLTg4MzItMDJkNjIxMGQx

User's query

In [6]:
user_query = "What resource should I refer to study for AI-900?"

Fetch the relevant chunk of data

In [7]:
retrieved_docs = vector_store.similarity_search(query=user_query, k=5)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

print(retrieved_docs)

Note

Access to this page requires authorization. You can try signing in or changing directories.

Access to this page requires authorization. You can try changing directories.

Study guide for Exam AI-900: Microsoft Azure AI Fundamentals

Purpose of this document

Warning

This exam will retire on June 30, 2026, at 11:59 PM Central Standard Time. Learn more.

This study guide should help you understand what to expect on the exam and includes a summary of the topics the exam might cover and links to additional resources. The information and materials in this document should help you focus your studies as you prepare for the exam.

Useful links Description How to earn the certification Some certifications only require passing one exam, while others require passing multiple exams. Certification renewal Microsoft associate, expert, and specialty certifications expire annually. You can renew by passing a free online assessment on Microsoft Learn. Your Microsoft Learn profile Connecting you

Create prompt template

In [8]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
       You are an expert Microsoft Azure cloud instructor and mentor. Your task is to suggest practical project ideas that align with specific Microsoft Azure certification objectives. Base your recommendations strictly on the study guide content provided.

       Study Guide Content: {relevant_chunk_of_data}

       User Question: {prompt}

       Instructions for your response:
        1. Provide 3–5 project ideas relevant to the user’s question and the study guide content.
        2. For each project, include:
          - A clear project title
          - A brief description (2–3 sentences)
          - Which Azure services and tools would be involved
          - How it relates to the certification objectives
        3. Do not include information not covered in the study guide.
        4. Keep your language beginner-friendly and actionable.

       Format your response as a numbered list for clarity.
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [9]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="gpt-4.1-mini",
    openai_api_version="2024-12-01-preview",
)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

print(response.content)

Certainly! Here are some practical project ideas to help you study and prepare for the AI-900: Microsoft Azure AI Fundamentals exam, based on the study guide content:

1. **Image Classification App Using Azure Computer Vision**  
   - **Description:** Build a simple web app that uploads images and classifies the objects or scenes within them using Azure’s Computer Vision service. This project will help you understand how computer vision works and the types of AI workloads it supports.  
   - **Azure Services:** Azure Computer Vision, Azure Blob Storage (to store images), Azure Web App or Azure Functions (for hosting the app).  
   - **Certification Objective:** Describes features of computer vision workloads on Azure (15–20%).

2. **Sentiment Analysis Tool with Azure Text Analytics**  
   - **Description:** Create a tool that accepts user comments or social media posts and analyzes their sentiment (positive, negative, neutral) using Azure Text Analytics. This will familiarize you with 